# PRUEBA CON S&P500

In [ ]:
!pip install yfinance pandas_ta PyPortfolioOpt PyPortfolioOpt ray pandas_datareader

In [1]:
# %%writefile RollingOLSRegression.py

from statsmodels.regression.rolling import RollingOLS
import pandas_datareader.data as web
import matplotlib.pyplot as plt
import statsmodels.api as sm
import pandas as pd
import numpy as np
from statsmodels.regression.rolling import RollingOLS
import pandas_datareader.data as web
import matplotlib.pyplot as plt
import statsmodels.api as sm
import pandas as pd
import numpy as np
import datetime as dt
import yfinance as yf
import pandas_ta
import warnings
import copy
from sklearn.cluster import KMeans
from pypfopt.efficient_frontier import EfficientFrontier
from pypfopt import risk_models
from pypfopt import expected_returns
import ray
import time

warnings.filterwarnings('ignore')

class RollingOLSRegression:

  def __init__(self,symbols_list,start_date,end_date):
    self.model=None;
    self.symbols_list=symbols_list;
    self.start_date=start_date;
    self.end_date=end_date;


  def load_data(self):
    data= yf.download(tickers=self.symbols_list,
                 start=self.start_date,
                 end=self.end_date,auto_adjust=False)
    return data

  #paralelizable
  def calculate_tecnical_indicators(self,data):
    df=copy.deepcopy(data)
    df=df.stack()
    df.index.names=['date','ticker']
    df.columns=df.columns.str.lower()


    # calculate Garman-Klass Volatility, rsi,bollinger bands, atr,macd,dollar volume
    df['garman_klass_vol'] = ((np.log(df['high'])-np.log(df['low']))**2)/2-(2*np.log(2)-1)*((np.log(df['adj close'])-np.log(df['open']))**2)

    df['rsi'] = df.groupby(level=1)['adj close'].transform(lambda x: pandas_ta.rsi(close=x, length=20))

    df['bb_low'] = df.groupby(level=1)['adj close'].transform(lambda x: pandas_ta.bbands(close=np.log1p(x), length=20).iloc[:,0])

    df['bb_mid'] = df.groupby(level=1)['adj close'].transform(lambda x: pandas_ta.bbands(close=np.log1p(x), length=20).iloc[:,1])

    df['bb_high'] = df.groupby(level=1)['adj close'].transform(lambda x: pandas_ta.bbands(close=np.log1p(x), length=20).iloc[:,2])

    def compute_atr(stock_data):
        atr = pandas_ta.atr(high=stock_data['high'],
                            low=stock_data['low'],
                            close=stock_data['close'],
                            length=14)
        return atr.sub(atr.mean()).div(atr.std())

    df['atr'] = df.groupby(level=1, group_keys=False).apply(compute_atr)

    def compute_macd(close):
        macd = pandas_ta.macd(close=close, length=20).iloc[:,0]
        return macd.sub(macd.mean()).div(macd.std())

    df['macd'] = df.groupby(level=1, group_keys=False)['adj close'].apply(compute_macd)

    df['dollar_volume'] = (df['adj close']*df['volume'])/1e6

    return df

  def aggregate_to_monthly_level(self,df):

    # To reduce training time and experiment with features and strategies,
    # we convert the business-daily data to month-end frequency.


    last_cols = [c for c in df.columns.unique(0) if c not in ['dollar_volume', 'volume', 'open',
                                                            'high', 'low', 'close']]

    data = (pd.concat([df.unstack('ticker')['dollar_volume'].resample('M').mean().stack('ticker').to_frame('dollar_volume'),
                    df.unstack()[last_cols].resample('M').last().stack('ticker')],
                    axis=1)).dropna()

    # Calculate 5-year rolling average of dollar volume for each stocks before filtering.

    data['dollar_volume'] = (data.loc[:, 'dollar_volume'].unstack('ticker').rolling(5*12, min_periods=12).mean().stack())

    data['dollar_vol_rank'] = (data.groupby('date')['dollar_volume'].rank(ascending=False))

    data = data[data['dollar_vol_rank']<150].drop(['dollar_volume', 'dollar_vol_rank'], axis=1)

    return data



  # parlelizable
  def Calculate_Montly_Returns(self,data):

    # To capture time series dynamics that reflect, for example,
    # momentum patterns, we compute historical returns using the method
    # .pct_change(lag), that is, returns over various monthly periods as identified by lags.

    def calculate_returns(df):

      outlier_cutoff = 0.005

      lags = [1, 2, 3, 6, 9, 12]

      for lag in lags:

          df[f'return_{lag}m'] = (df['adj close']
                                .pct_change(lag)
                                .pipe(lambda x: x.clip(lower=x.quantile(outlier_cutoff),
                                                      upper=x.quantile(1-outlier_cutoff)))
                                .add(1)
                                .pow(1/lag)
                                .sub(1))
      return df


    data = data.groupby(level=1, group_keys=False).apply(calculate_returns).dropna()

    return data

  #paralizable
  def download_fama_f_and_calculate_rolling_f_b(self,data):
      factor_data = web.DataReader('F-F_Research_Data_5_Factors_2x3',
                               'famafrench',
                               start='2010')[0].drop('RF', axis=1)

      factor_data.index = factor_data.index.to_timestamp()

      factor_data = factor_data.resample('M').last().div(100)

      factor_data.index.name = 'date'

      factor_data = factor_data.join(data['return_1m']).sort_index()

      # filter out stocks whit less than 10 months of data

      observations = factor_data.groupby(level=1).size()

      valid_stocks = observations[observations >= 10]

      factor_data = factor_data[factor_data.index.get_level_values('ticker').isin(valid_stocks.index)]

      #calculate rolling factor betas

      betas = (factor_data.groupby(level=1,
                            group_keys=False)
         .apply(lambda x: RollingOLS(endog=x['return_1m'],
                                     exog=sm.add_constant(x.drop('return_1m', axis=1)),
                                     window=min(24, x.shape[0]),
                                     min_nobs=len(x.columns)+1)
         .fit(params_only=True)
         .params
         .drop('const', axis=1)))

      return betas

  def join_the_rolling_factors_data(self,betas,data):
      factors = ['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA']

      data = (data.join(betas.groupby('ticker').shift()))

      data.loc[:, factors] = data.groupby('ticker', group_keys=False)[factors].apply(lambda x: x.fillna(x.mean()))

      data = data.drop('adj close', axis=1)

      data = data.dropna()

      return data

  def apply_pre_defined_centroids(self):

    target_rsi_values = [30, 45, 55, 70]

    initial_centroids = np.zeros((len(target_rsi_values), 18))

    initial_centroids[:, 6] = target_rsi_values

    return initial_centroids

  def k_means_clustering(self,data,initial_centroids):
    def get_clusters(df):
      df['cluster'] = KMeans(n_clusters=4,
                            random_state=0,
                            init=initial_centroids).fit(df).labels_
      return df

    data = data.dropna().groupby('date', group_keys=False).apply(get_clusters)

    return data

  def portfolio_based__on_cluster(self,data):
    filtered_df = data[data['cluster']==3].copy()

    filtered_df = filtered_df.reset_index(level=1)

    filtered_df.index = filtered_df.index+pd.DateOffset(1)

    filtered_df = filtered_df.reset_index().set_index(['date', 'ticker'])

    dates = filtered_df.index.get_level_values('date').unique().tolist()

    fixed_dates = {}

    for d in dates:

        fixed_dates[d.strftime('%Y-%m-%d')] = filtered_df.xs(d, level=0).index.tolist()

    return fixed_dates


  def optimize_weights(prices, lower_bound=0):

    returns = expected_returns.mean_historical_return(prices=prices,
                                                      frequency=252)

    cov = risk_models.sample_cov(prices=prices,
                                 frequency=252)

    ef = EfficientFrontier(expected_returns=returns,
                           cov_matrix=cov,
                           weight_bounds=(lower_bound, .1),
                           solver='SCS')

    weights = ef.max_sharpe()

    return ef.clean_weights()

  #paralelizable
  def download_fresh_daily_prices(self,data):
    stocks = data.index.get_level_values('ticker').unique().tolist()

    new_df = yf.download(tickers=stocks,
                        start=data.index.get_level_values('date').unique()[0]-pd.DateOffset(months=12),
                        end=data.index.get_level_values('date').unique()[-1],auto_adjust=False)

    return new_df
  #paralelizable
  def calculate_each_day_portfolio_return(self,new_df,fixed_dates):

    returns_dataframe = np.log(new_df['Adj Close']).diff()
    
    portfolio_df = pd.DataFrame()
    
    for start_date in fixed_dates.keys():
    
        try:
            end_date = (pd.to_datetime(start_date)+pd.offsets.MonthEnd(0)).strftime('%Y-%m-%d')
            cols = fixed_dates[start_date]
            optimization_start_date = (pd.to_datetime(start_date)-pd.DateOffset(months=12)).strftime('%Y-%m-%d')
            optimization_end_date = (pd.to_datetime(start_date)-pd.DateOffset(days=1)).strftime('%Y-%m-%d')
    
            optimization_df = new_df[optimization_start_date:optimization_end_date]['Adj Close'][cols]
    
            success = False
            try:
                weights = optimize_weights(prices=optimization_df,
                                       lower_bound=round(1/(len(optimization_df.columns)*2),3))
                weights = pd.DataFrame(weights, index=optimization_df.columns, columns=[0])
                success = True
            except:
                print(f'Max Sharpe Optimization failed for {start_date}, Continuing with Equal-Weights')
    
            if success==False:
                weights = pd.DataFrame([1/len(optimization_df.columns) for i in range(len(optimization_df.columns))],
                                         index=optimization_df.columns.tolist(),
                                         columns=[0])
    
            temp_df = returns_dataframe[start_date:end_date][cols]
    
            # Create portfolio returns by calculating weighted average
            portfolio_returns = []
            for date in temp_df.index:
                daily_return = 0
                for ticker in temp_df.columns:
                    if pd.notna(temp_df.loc[date, ticker]) and ticker in weights.index:
                        daily_return += temp_df.loc[date, ticker] * weights.loc[ticker, 0]
                portfolio_returns.append(daily_return)
    
            temp_portfolio_df = pd.DataFrame(portfolio_returns,
                                           index=temp_df.index,
                                           columns=['Strategy Return'])
    
            portfolio_df = pd.concat([portfolio_df, temp_portfolio_df], axis=0)
    
        except Exception as e:
            print(f"Error for {start_date}: {e}")
    
    portfolio_df = portfolio_df.drop_duplicates()
    
    return portfolio_df


  def get_portfolio_returns_to_visualize(self,ticker,start_date,end_date,portfolio_df):

    data= yf.download(tickers=ticker,
                  start=start_date,
                  end=end_date, auto_adjust=False)
    data.columns = data.columns.droplevel(1)

    data_ret = np.log(data[['Adj Close']]).diff().dropna().rename({'Adj Close':f'{ticker} Buy&Hold'}, axis=1)

    portfolio_df = portfolio_df.merge(data_ret,
                                  left_index=True,
                                  right_index=True)
    return portfolio_df




In [2]:
#descarga sequencial

end_date = '2023-09-27'

start_date = pd.to_datetime(end_date)-pd.DateOffset(365*8)

sp500 = pd.read_html('https://en.wikipedia.org/wiki/List_of_S%26P_500_companies')[0]

sp500['Symbol'] = sp500['Symbol'].str.replace('.', '-')

symbols_list = sp500['Symbol'].unique().tolist()

symbols_list=list(filter(lambda x: x not in ['SOLV', 'GEV', 'SW', 'VLTO'],symbols_list))

start_time=time.time()

df_sequential= yf.download(tickers=symbols_list,
              start=start_date,
              end=end_date,auto_adjust=False)

print(f'df_sequential time: {time.time()-start_time}')




[*********************100%***********************]  499 of 499 completed


df_sequential time: 37.42143106460571


In [ ]:
# descarga paralela

@ray.remote(num_cpus=2)
def download_data_by_tickers(stocks,start,end):
    new_df = yf.download(tickers=stocks,
                        start=start,
                        end=end,auto_adjust=False)

    return new_df



batch_size=10

ticker_batches=np.array_split(symbols_list,batch_size)

start_time=time.time()

futures=[download_data_by_tickers.remote(stock.tolist(),start_date,end_date) for stock in ticker_batches]

results=ray.get(futures)

df_parallel=pd.concat(results,axis=1)

print(f'df_parallel time: {time.time()-start_time}')





# SI SIRVIO


In [ ]:
# %%writefile profiling.py

# import cProfile
# import time
# from RollingOLSRegression import RollingOLSRegression
# import pandas as pd
# import datetime as dt


def profiling_pipeline():
    end_date = '2023-09-27'

    start_date = pd.to_datetime(end_date)-pd.DateOffset(365*8)
    
    sp500 = pd.read_html('https://en.wikipedia.org/wiki/List_of_S%26P_500_companies')[0]
    
    sp500['Symbol'] = sp500['Symbol'].str.replace('.', '-')
    
    symbols_list = sp500['Symbol'].unique().tolist()
    
    symbols_list=list(filter(lambda x: x not in ['SOLV', 'GEV', 'SW', 'VLTO'],symbols_list))

    rolling_ols = RollingOLSRegression(symbols_list,start_date,end_date)
    
    start_time = time.time()
    df= rolling_ols.load_data()
    print(f"Data Loading Time: {time.time() - start_time} seconds")
    
    
    start_time = time.time()
    df_indicators = rolling_ols.calculate_tecnical_indicators(df)
    
    df_aggregate= rolling_ols.aggregate_to_monthly_level(df_indicators)
    
    df_montly_returns = rolling_ols.Calculate_Montly_Returns(df_aggregate)
    
    betas = rolling_ols.download_fama_f_and_calculate_rolling_f_b(df_montly_returns)
    
    join_data=rolling_ols.join_the_rolling_factors_data(betas,df_montly_returns)
    
    centroids=rolling_ols.apply_pre_defined_centroids()
    
    # print(f'preprocess secuential data: {time.time()-start_time}')
    
    # start_time=time.time()
    
    # # data_with_clusters = rolling_ols.k_means_clustering(join_data,centroids)
    
    # # fixed_dates = rolling_ols.portfolio_based__on_cluster(data_with_clusters)
    
    clusters=rolling_ols.k_means_clustering(join_data,centroids)
    
    fixed_dates = rolling_ols.portfolio_based__on_cluster(clusters)
    
    # print(f'clustering time: {time.time()-start_time}')
    
    
    # start_time=time.time()
    fresh_daily_prices=rolling_ols.download_fresh_daily_prices(clusters)
    
    portfolio_returns=rolling_ols.calculate_each_day_portfolio_return(fresh_daily_prices,fixed_dates)
    
    start_date='2015-01-01'
    
    end_date=dt.date.today()
    
    final_portfolio=rolling_ols.get_portfolio_returns_to_visualize('SPY',start_date,end_date,portfolio_returns)
    
    # print(f'final portfolio inference time :{time.time()-start_time}')

    return final_portfolio


# if __name__ == "__main__":
#   cProfile.run("profiling_pipeline()")

final_portfolio=profiling_pipeline()

final_portfolio



In [4]:
#depuracion

# end_date = '2023-09-27'

# start_date = pd.to_datetime(end_date)-pd.DateOffset(365*8)

# sp500 = pd.read_html('https://en.wikipedia.org/wiki/List_of_S%26P_500_companies')[0]

# sp500['Symbol'] = sp500['Symbol'].str.replace('.', '-')

# symbols_list = sp500['Symbol'].unique().tolist()

# symbols_list=list(filter(lambda x: x not in ['SOLV', 'GEV', 'SW', 'VLTO'],symbols_list))

# rolling=RollingOLSRegression(symbols_list,start_date,end_date)

# df=rolling.load_data()

# df_indicators = rolling.calculate_tecnical_indicators(df)

# df_aggregate= rolling.aggregate_to_monthly_level(df_indicators)

# df_montly_returns = rolling.Calculate_Montly_Returns(df_aggregate)

# betas = rolling.download_fama_f_and_calculate_rolling_f_b(df_montly_returns)

# join_data=rolling.join_the_rolling_factors_data(betas,df_montly_returns)

# centroids=rolling.apply_pre_defined_centroids()

# clusters=rolling.k_means_clustering(join_data,centroids)

# fixed_dates = rolling.portfolio_based__on_cluster(clusters)

# fresh_daily_prices=rolling.download_fresh_daily_prices(clusters)

# portfolio_returns=rolling.calculate_each_day_portfolio_return(fresh_daily_prices,fixed_dates)

# start_date='2015-01-01'
    
# end_date=dt.date.today()

# final_portfolio=rolling.get_portfolio_returns_to_visualize('SPY',start_date,end_date,portfolio_returns)


# final_portfolio


# portfolio_returns


Max Sharpe Optimization failed for 2017-11-01, Continuing with Equal-Weights
Max Sharpe Optimization failed for 2017-12-01, Continuing with Equal-Weights
Max Sharpe Optimization failed for 2018-01-01, Continuing with Equal-Weights
Max Sharpe Optimization failed for 2018-02-01, Continuing with Equal-Weights
Max Sharpe Optimization failed for 2018-03-01, Continuing with Equal-Weights
Max Sharpe Optimization failed for 2018-04-01, Continuing with Equal-Weights
Max Sharpe Optimization failed for 2018-05-01, Continuing with Equal-Weights
Max Sharpe Optimization failed for 2018-06-01, Continuing with Equal-Weights
Max Sharpe Optimization failed for 2018-07-01, Continuing with Equal-Weights
Max Sharpe Optimization failed for 2018-08-01, Continuing with Equal-Weights
Max Sharpe Optimization failed for 2018-09-01, Continuing with Equal-Weights
Max Sharpe Optimization failed for 2018-10-01, Continuing with Equal-Weights
Max Sharpe Optimization failed for 2018-11-01, Continuing with Equal-Weights

,Strategy Return
Date,
2017-11-01,0.000292
2017-11-02,0.002176
2017-11-03,0.000726
2017-11-06,-0.001573
2017-11-07,-0.004801
...,...
2023-09-25,0.003047
2023-09-26,-0.014167
2023-09-27,0.005061


In [6]:
# funcion auxiliar para partir en lotes

def split_df_by_tickers(df, batch_size=10):
    # Get unique tickers from level 1
    tickers = df.columns.get_level_values(1).unique()
    # print(tickers)

    
    
    # Split tickers into batches
    # ticker_batches = [tickers[i:i + batch_size] for i in range(0, len(tickers), batch_size)]
    ticker_batches=np.array_split(tickers,batch_size)
   
    
    # Create DataFrames for each batch
    df_batches = []
    for batch_tickers in ticker_batches:
        # Select columns for current batch of tickers
        batch_df = df.loc[:, (slice(None), batch_tickers)]
        df_batches.append(batch_df)
    
    return df_batches



## PARALELIZACION DE calulate_tecnical_indicators

In [4]:

# @ray.remote(num_cpus=2)
def calculate_tecnical_indicators(df):
    df=copy.deepcopy(df)
    df=df.stack()
    df.index.names=['date','ticker']
    df.columns=df.columns.str.lower()


    # calculate Garman-Klass Volatility, rsi,bollinger bands, atr,macd,dollar volume
    df['garman_klass_vol'] = ((np.log(df['high'])-np.log(df['low']))**2)/2-(2*np.log(2)-1)*((np.log(df['adj close'])-np.log(df['open']))**2)

    df['rsi'] = df.groupby(level=1)['adj close'].transform(lambda x: pandas_ta.rsi(close=x, length=20))

    df['bb_low'] = df.groupby(level=1)['adj close'].transform(lambda x: pandas_ta.bbands(close=np.log1p(x), length=20).iloc[:,0])

    df['bb_mid'] = df.groupby(level=1)['adj close'].transform(lambda x: pandas_ta.bbands(close=np.log1p(x), length=20).iloc[:,1])

    df['bb_high'] = df.groupby(level=1)['adj close'].transform(lambda x: pandas_ta.bbands(close=np.log1p(x), length=20).iloc[:,2])

    def compute_atr(stock_data):
        atr = pandas_ta.atr(high=stock_data['high'],
                            low=stock_data['low'],
                            close=stock_data['close'],
                            length=14)
        return atr.sub(atr.mean()).div(atr.std())

    df['atr'] = df.groupby(level=1, group_keys=False).apply(compute_atr)

    def compute_macd(close):
        macd = pandas_ta.macd(close=close, length=20).iloc[:,0]
        return macd.sub(macd.mean()).div(macd.std())

    df['macd'] = df.groupby(level=1, group_keys=False)['adj close'].apply(compute_macd)

    df['dollar_volume'] = (df['adj close']*df['volume'])/1e6


    return df




# batches=split_df_by_tickers(df_parallel,10)

# start_time=time.time();

# futures=[calculate_tecnical_indicators.remote(df) for df in batches]

# results = ray.get(futures)

# df_indicators_parallel=pd.concat(results)

# end_time=time.time()

# print(f'ray time: {end_time-start_time}')

# df_indicators_parallel










## VERSION SECUENCIAL  calculate_tecnical_indicators

In [5]:
#hay que comentar las partes de ray en la celda anterior

start_time=time.time();

df_indicators_seq = calculate_tecnical_indicators(df_sequential)

end_time=time.time()

print(f'secuential time: {end_time-start_time}')

df_indicators_seq

#SI SIRVIO PARALELIZAR

secuential time: 10.015934228897095


Price               adj close       close        high         low        open  \
date       ticker                                                               
2015-09-29 A        31.251007   33.740002   34.060001   33.240002   33.360001   
           AAPL     24.536386   27.264999   28.377501   26.965000   28.207500   
           ABBV     35.061207   52.790001   54.189999   51.880001   53.099998   
           ABT      32.820751   39.500000   40.150002   39.029999   39.259998   
           ACGL     23.217773   24.416668   24.456667   24.100000   24.170000   
...                       ...         ...         ...         ...         ...   
2023-09-26 XYL      87.701073   89.519997   90.849998   89.500000   90.379997   
           YUM     119.860718  124.010002  124.739998  123.449997  124.239998   
           ZBH     110.800156  112.459999  117.110001  112.419998  116.769997   
           ZBRA    223.960007  223.960007  226.649994  222.580002  225.970001   
           ZTS     173.604691  176.869995  178.449997  176.270004  176.580002   

Price                   volume  garman_klass_vol        rsi    bb_low  \
date       ticker                                                       
2015-09-29 A         2252400.0         -0.001351        NaN       NaN   
           AAPL    293461600.0         -0.006207        NaN       NaN   
           ABBV     12842800.0         -0.065607        NaN       NaN   
           ABT      12287500.0         -0.011997        NaN       NaN   
           ACGL      1888800.0         -0.000516        NaN       NaN   
...                        ...               ...        ...       ...   
2023-09-26 XYL       1322400.0         -0.000238  26.146751  4.474158   
           YUM       1500600.0         -0.000443  36.057210  4.792448   
           ZBH       3610500.0         -0.000229  31.893220  4.739333   
           ZBRA       355400.0          0.000133  29.494977  5.400991   
           ZTS       1463200.0         -0.000036  42.623461  5.145043   

Price                bb_mid   bb_high       atr      macd  dollar_volume  
date       ticker                                                         
2015-09-29 A            NaN       NaN       NaN       NaN      70.389768  
           AAPL         NaN       NaN       NaN       NaN    7200.487238  
           ABBV         NaN       NaN       NaN       NaN     450.284067  
           ABT          NaN       NaN       NaN       NaN     403.284980  
           ACGL         NaN       NaN       NaN       NaN      43.853730  
...                     ...       ...       ...       ...            ...  
2023-09-26 XYL     4.556072  4.637985  0.033800 -2.159189     115.975899  
           YUM     4.822408  4.852369  0.142547 -1.363695     179.862993  
           ZBH     4.778997  4.818662 -0.381708 -0.881067     400.043962  
           ZBRA    5.539167  5.677342 -0.057389 -1.600791      79.595386  
           ZTS     5.203852  5.262662  0.651515 -1.188278     254.018383  

[980418 rows x 14 columns]

In [61]:
num_na_ind_seq=df_indicators_seq.isna().sum()
num_na_ind_seq=num_na_ind_seq[num_na_ind_seq>0]

num_na_ind_seq

Price
rsi         9980
bb_low      9481
bb_mid      9481
bb_high     9481
atr         6986
macd       12475
dtype: int64

In [60]:
num_na_ind_parallel=df_indicators_parallel.isna().sum()
num_na_ind_parallel=num_na_ind_parallel[num_na_ind_parallel>0]

num_na_ind_parallel

Price
rsi         9980
bb_low      9481
bb_mid      9481
bb_high     9481
atr         6986
macd       12475
dtype: int64

## FUNCION SECUENCIAL AGGREGATE_TO_MONTLY_LEVEL(no es conveniente paralelizar)

In [6]:
def aggregate_to_monthly_level(df):

    # To reduce training time and experiment with features and strategies,
    # we convert the business-daily data to month-end frequency.


    last_cols = [c for c in df.columns.unique(0) if c not in ['dollar_volume', 'volume', 'open',
                                                            'high', 'low', 'close']]

    data = (pd.concat([df.unstack('ticker')['dollar_volume'].resample('M').mean().stack('ticker').to_frame('dollar_volume'),
                    df.unstack()[last_cols].resample('M').last().stack('ticker')],
                    axis=1)).dropna()

    # Calculate 5-year rolling average of dollar volume for each stocks before filtering.

    data['dollar_volume'] = (data.loc[:, 'dollar_volume'].unstack('ticker').rolling(5*12, min_periods=12).mean().stack())

    data['dollar_vol_rank'] = (data.groupby('date')['dollar_volume'].rank(ascending=False))

    data = data[data['dollar_vol_rank']<150].drop(['dollar_volume', 'dollar_vol_rank'], axis=1)

    return data

df_aggregate=aggregate_to_monthly_level(df_indicators_seq)

df_aggregate

adj close  garman_klass_vol        rsi    bb_low  \
date       ticker                                                      
2016-10-31 AAPL     26.090460         -0.002767  49.891104  3.289745   
           ABBV     38.834366         -0.056807  27.477698  3.718614   
           ABT      33.619492         -0.009785  38.008845  3.535356   
           ACN     101.760155         -0.006263  53.823629  4.619889   
           ADBE    107.510002          0.000059  53.668389  4.679513   
...                       ...               ...        ...       ...   
2023-09-30 VZ       29.285009         -0.005147  42.222456  3.404462   
           WDAY    229.240005          0.000141  43.976804  5.437299   
           WFC      38.847347         -0.000904  40.920294  3.673898   
           WMT      53.052738         -0.000186  54.722541  3.974950   
           XOM     109.552101         -0.001011  59.440202  4.653137   

                     bb_mid   bb_high       atr      macd  
date       ticker                                          
2016-10-31 AAPL    3.318620  3.347495 -1.038688 -0.195978  
           ABBV    3.772733  3.826853 -0.893132 -0.760594  
           ABT     3.585802  3.636249 -1.035224 -0.650888  
           ACN     4.631524  4.643160 -0.996806 -0.135456  
           ADBE    4.694639  4.709766 -1.230331 -0.109039  
...                     ...       ...       ...       ...  
2023-09-30 VZ      3.436605  3.468747 -1.078816 -0.350386  
           WDAY    5.495242  5.553185 -0.127668 -0.306858  
           WFC     3.714239  3.754579 -0.558742 -0.282324  
           WMT     3.992416  4.009883 -0.196381  0.399460  
           XOM     4.693220  4.733304  0.601335  1.400623  

[12516 rows x 8 columns]

## Version Paralela calculate_montly_returns

In [9]:
# parlelizable

# @ray.remote(num_cpus=2)
def Calculate_Montly_Returns(data):

  # To capture time series dynamics that reflect, for example,
  # momentum patterns, we compute historical returns using the method
  # .pct_change(lag), that is, returns over various monthly periods as identified by lags.
    
  data=copy.deepcopy(data)
  # data=data.stack()
  # data.index.names=['date','ticker']
  # data.columns=data.columns.str.lower()

  def calculate_returns(df):

    outlier_cutoff = 0.005

    lags = [1, 2, 3, 6, 9, 12]

    for lag in lags:

        df[f'return_{lag}m'] = (df['adj close']
                              .pct_change(lag)
                              .pipe(lambda x: x.clip(lower=x.quantile(outlier_cutoff),
                                                    upper=x.quantile(1-outlier_cutoff)))
                              .add(1)
                              .pow(1/lag)
                              .sub(1))
    return df


  data = data.groupby(level=1, group_keys=False).apply(calculate_returns).dropna()

  return data



# df= df_aggregate.unstack()

# batches=split_df_by_tickers(df,10)

# start_date=time.time()

# futures=[Calculate_Montly_Returns.remote(df) for df in batches]

# results=ray.get(futures)

# montly_returns=pd.concat(results)

# end_date=time.time()

# print(f'calculate_Montly_returns parallel time: {end_date-start_date}')

# montly_returns




## Version Secuencial calculate_montly_returns()

In [10]:
start_time=time.time()

montly_returns=Calculate_Montly_Returns(df_aggregate)

end_time=time.time()

print(f'calculate_Montly_returns secuencial time: {end_time-start_time}')

montly_returns



calculate_Montly_returns secuencial time: 1.9544148445129395


adj close  garman_klass_vol        rsi    bb_low  \
date       ticker                                                      
2017-10-31 AAPL     39.529045         -0.001263  69.196689  3.590192   
           ABBV     65.125313         -0.042703  55.247898  4.161565   
           ABT      47.540340         -0.007128  53.844900  3.873128   
           ACN     127.139015         -0.005423  69.365354  4.785195   
           ADBE    175.160004          0.000067  70.089317  4.951759   
...                       ...               ...        ...       ...   
2023-09-30 VZ       29.285009         -0.005147  42.222456  3.404462   
           WDAY    229.240005          0.000141  43.976804  5.437299   
           WFC      38.847347         -0.000904  40.920294  3.673898   
           WMT      53.052738         -0.000186  54.722541  3.974950   
           XOM     109.552101         -0.001011  59.440202  4.653137   

                     bb_mid   bb_high       atr      macd  return_1m  \
date       ticker                                                      
2017-10-31 AAPL    3.637059  3.683926 -0.906642 -0.039275   0.096808   
           ABBV    4.207901  4.254238  0.375557  0.473815   0.022728   
           ABT     3.896688  3.920248 -1.040044  0.276133   0.021276   
           ACN     4.824869  4.864543 -0.986514  0.352343   0.064180   
           ADBE    5.089292  5.226825 -0.888269  0.612102   0.174152   
...                     ...       ...       ...       ...        ...   
2023-09-30 VZ      3.436605  3.468747 -1.078816 -0.350386  -0.056890   
           WDAY    5.495242  5.553185 -0.127668 -0.306858  -0.062413   
           WFC     3.714239  3.754579 -0.558742 -0.282324  -0.015500   
           WMT     3.992416  4.009883 -0.196381  0.399460  -0.000676   
           XOM     4.693220  4.733304  0.601335  1.400623   0.046947   

                   return_2m  return_3m  return_6m  return_9m  return_12m  
date       ticker                                                          
2017-10-31 AAPL     0.015250   0.044955   0.028875   0.038941    0.035228  
           ABBV     0.098590   0.091379   0.056495   0.047273    0.044026  
           ABT      0.034308   0.034801   0.038672   0.031320    0.029294  
           ACN      0.048455   0.037203   0.028692   0.027398    0.018728  
           ADBE     0.062497   0.061392   0.045993   0.049515    0.041515  
...                      ...        ...        ...        ...         ...  
2023-09-30 VZ      -0.016122  -0.033458  -0.021495  -0.014100   -0.006158  
           WDAY    -0.016777   0.004919   0.017531   0.035597    0.034709  
           WFC     -0.057917  -0.013554   0.016712   0.000703    0.003255  
           WMT      0.010014   0.012354   0.017574   0.016553    0.020256  
           XOM      0.046139   0.030496   0.012838   0.008747    0.027037  

[10340 rows x 14 columns]

## PARALELIZACION DEL PROCESO DE ENTRENAMIENTO

## funciones que no se paralelizan

In [13]:
def download_fama_french_F(montly_returns,):
    data=montly_returns
    factor_data = web.DataReader('F-F_Research_Data_5_Factors_2x3',
                               'famafrench',
                               start='2010')[0].drop('RF', axis=1)

    factor_data.index = factor_data.index.to_timestamp()
    
    factor_data = factor_data.resample('M').last().div(100)
    
    factor_data.index.name = 'date'
    
    factor_data = factor_data.join(data['return_1m']).sort_index()
    
    observations = factor_data.groupby(level=1).size()

    valid_stocks = observations[observations >= 10]
    
    factor_data = factor_data[factor_data.index.get_level_values('ticker').isin(valid_stocks.index)]
    
    return factor_data


def calculate_rolling_f_betas(factor_data):
    betas = (factor_data.groupby(level=1,
                            group_keys=False)
         .apply(lambda x: RollingOLS(endog=x['return_1m'], 
                                     exog=sm.add_constant(x.drop('return_1m', axis=1)),
                                     window=min(24, x.shape[0]),
                                     min_nobs=len(x.columns)+1)
         .fit(params_only=True)
         .params
         .drop('const', axis=1)))

    return betas

def join_r_factors_to_main_features(montly_returns):

    data=montly_returns
    
    factors = ['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA']

    data = (data.join(betas.groupby('ticker').shift()))
    
    data.loc[:, factors] = data.groupby('ticker', group_keys=False)[factors].apply(lambda x: x.fillna(x.mean()))
    
    data = data.drop('adj close', axis=1)
    
    data = data.dropna()
    
    return data
    
    




def apply_pre_defined_centroids_kmeans(join_data):

    data=join_data

    target_rsi_values = [30, 45, 55, 70]

    initial_centroids = np.zeros((len(target_rsi_values), 18))

    initial_centroids[:, 6] = target_rsi_values


    #k_means_clustering
    def get_clusters(df):
      df['cluster'] = KMeans(n_clusters=4,
                            random_state=0,
                            init=initial_centroids).fit(df).labels_
      return df

    data = data.dropna().groupby('date', group_keys=False).apply(get_clusters)

    return data


def portfolio_based__on_cluster(cluster_data):
    
    data=cluster_data
    
    filtered_df = data[data['cluster']==3].copy()

    filtered_df = filtered_df.reset_index(level=1)

    filtered_df.index = filtered_df.index+pd.DateOffset(1)

    filtered_df = filtered_df.reset_index().set_index(['date', 'ticker'])

    dates = filtered_df.index.get_level_values('date').unique().tolist()

    fixed_dates = {}

    for d in dates:

        fixed_dates[d.strftime('%Y-%m-%d')] = filtered_df.xs(d, level=0).index.tolist()

    return fixed_dates

def optimize_weights(prices, lower_bound=0):

    returns = expected_returns.mean_historical_return(prices=prices,
                                                      frequency=252)

    cov = risk_models.sample_cov(prices=prices,
                                 frequency=252)

    ef = EfficientFrontier(expected_returns=returns,
                           cov_matrix=cov,
                           weight_bounds=(lower_bound, .1),
                           solver='SCS')

    weights = ef.max_sharpe()

    return ef.clean_weights()

def download_fresh_daily_prices(data):
    stocks = data.index.get_level_values('ticker').unique().tolist()

    new_df = yf.download(tickers=stocks,
                        start=data.index.get_level_values('date').unique()[0]-pd.DateOffset(months=12),
                        end=data.index.get_level_values('date').unique()[-1],auto_adjust=False)

    return new_df

In [14]:
#pipeline funciones anteriores


start_time=time.time()
factor_data=download_fama_french_F(montly_returns) #mas pesados
print(f'factor_data time: {time.time()-start_time}')

start_time=time.time()
betas=calculate_rolling_f_betas(factor_data) #mas pesados
print(f'betas time: {time.time()-start_time}')

start_time=time.time()
join_data=join_r_factors_to_main_features(montly_returns) #mas pesados
print(f'join_data time: {time.time()-start_time}')

start_time=time.time()
clusters_data=apply_pre_defined_centroids_kmeans(join_data)
print(f'clusters_data time:{time.time()-start_time}')

start_time=time.time()
fixed_dates=portfolio_based__on_cluster(clusters_data)
print(f'fixed_dates time:{time.time()-start_time}')








factor_data time: 0.7466566562652588
betas time: 0.6743857860565186
join_data time: 0.21073222160339355
clusters_data time:0.18568730354309082
fixed_dates time:0.021031618118286133


## PARALELIZACION DE CALCULATE_ROLLING_F_BETAS

In [15]:
#pruebas

@ray.remote(num_cpus=2)
def calculate_rolling_f_betas(factor_data):
    factor_data=factor_data.stack()
    betas = (factor_data.groupby(level=1,
                            group_keys=False)
         .apply(lambda x: RollingOLS(endog=x['return_1m'], 
                                     exog=sm.add_constant(x.drop('return_1m', axis=1)),
                                     window=min(24, x.shape[0]),
                                     min_nobs=len(x.columns)+1)
         .fit(params_only=True)
         .params
         .drop('const', axis=1)))

    return betas


fd=fd.unstack()

factor_batches=split_df_by_tickers(fd,10)

start_time=time.time()
futures=[calculate_rolling_f_betas.remote(fb) for fb in factor_batches]

results=ray.get(futures)

betas=pd.concat(results)

print(f'calculate_rolling_f_betas parallel time: {time.time()-start_time}')

betas


calculate_rolling_f_betas parallel time: 5.744250535964966


Mkt-RF       SMB       HML        RMW       CMA
date       ticker                                                   
2017-10-31 AAPL         NaN       NaN       NaN        NaN       NaN
           ABBV         NaN       NaN       NaN        NaN       NaN
           ABT          NaN       NaN       NaN        NaN       NaN
           ACN          NaN       NaN       NaN        NaN       NaN
           ADBE         NaN       NaN       NaN        NaN       NaN
...                     ...       ...       ...        ...       ...
2023-09-30 NEM     0.527419 -0.516512 -0.760532  -0.650831  2.140848
           PLTR    3.036109 -5.003911  2.635989 -11.081319 -3.117794
           RCL     2.172724  1.166065  0.224789  -1.388278  0.544531
           SPGI    1.083316  0.398716 -0.842262   0.681783  0.568334
           UBER    1.137701  1.148945 -0.438119  -1.300690 -0.341582

[10310 rows x 5 columns]

In [19]:
# version sequencial calculate_rolling_f_betas

start_time=time.time()

d=calculate_rolling_f_betas(factor_data)

print(f'calculate_rolling_f_betas sequential: {time.time()-start_time}')

d

#SI SIRVIO PARALELIZAR

calculate_rolling_f_betas sequential: 0.9936761856079102


Mkt-RF       SMB       HML       RMW       CMA
date       ticker                                                  
2017-10-31 AAPL         NaN       NaN       NaN       NaN       NaN
           ABBV         NaN       NaN       NaN       NaN       NaN
           ABT          NaN       NaN       NaN       NaN       NaN
           ACN          NaN       NaN       NaN       NaN       NaN
           ADBE         NaN       NaN       NaN       NaN       NaN
...                     ...       ...       ...       ...       ...
2023-09-30 VZ      0.332870 -0.161915  0.271797  0.322040  0.101615
           WDAY    1.079640 -0.947248 -0.557469 -0.906769 -0.250535
           WFC     1.112738  0.257062  2.040613 -0.472583 -1.546792
           WMT     0.704617 -0.316998 -0.400224 -0.143194  0.497980
           XOM     0.974994 -1.118671  1.720715 -0.655853 -0.362729

[10310 rows x 5 columns]

# Paralelizacion download_fresh_daily_prices_data

In [16]:

@ray.remote(num_cpus=2)
def download_fresh_daily_prices(stocks,start,end):
    # stocks = data.index.get_level_values('ticker').unique().tolist()

    new_df = yf.download(tickers=stocks,
                        start=start,
                        end=end,auto_adjust=False)

    return new_df

data=clusters_data.stack()

start=data.index.get_level_values('date').unique()[0]-pd.DateOffset(months=12)
end=data.index.get_level_values('date').unique()[-1]

stocks = data.index.get_level_values('ticker').unique().tolist()

ticker_batches=np.array_split(stocks,10)

start_time=time.time()

futures=[download_fresh_daily_prices.remote(batch.tolist(),start,end) for batch in ticker_batches]

results=ray.get(futures)

print(f'download_fresh_daily_prices time:{time.time()-start_time}')

new_df=pd.concat(results,axis=1)

new_df


[******                12%                       ]  2 of 16 completed
[*********             19%                       ]  3 of 16 completed
[                       0%                       ]
[***************       31%                       ]  5 of 16 completed
[*********             19%                       ]  3 of 16 completed
[******                12%                       ]  2 of 16 completed
[************                                    ]
[***************       25%                       ]  5 of 16 completed  5 of 16 completed
[******************    38%                       ]  6 of 16 completed
[******************    38%                       ]  6 of 16 completed
[********************* 44%                       ]  7 of 16 completed
[************          25%                       ]  4 of 16 completed
[***************       31%                       ]  5 of 16 completed
[**********************50%                       ]  8 of 16 completed
[**********************62%*****        

download_fresh_daily_prices time:11.970920324325562


Price        Adj Close                                                 \
Ticker            AAPL        ABBV        ABT         ACN        ADBE   
Date                                                                    
2016-10-31   26.090454   38.834370  33.619492  101.760155  107.510002   
2016-11-01   25.619383   39.300819  33.456696  101.672630  106.870003   
2016-11-02   25.642363   39.537537  33.156834  103.834946  105.889999   
2016-11-03   25.367516   38.910950  32.882671  102.381714  107.169998   
2016-11-04   25.138840   39.015385  33.490974  102.626854  106.199997   
...                ...         ...        ...         ...         ...   
2023-09-25  174.569946  144.997543  94.104805  307.845001  511.600006   
2023-09-26  170.485275  144.303757  92.907623  301.649078  506.299988   
2023-09-27  168.968384  143.572433  92.231773  305.310272  502.600006   
2023-09-28  169.226166  142.747345  94.732368  292.092896  504.670013   
2023-09-29  169.741699  139.756454  93.506210  298.250031  509.899994   

Price                                                                 ...  \
Ticker            AIG        AMAT        AMGN        AMZN       AVGO  ...   
Date                                                                  ...   
2016-10-31  49.670326   26.357296  108.539726   39.491001  13.219683  ...   
2016-11-01  48.849197   26.194155  108.232132   39.270500  13.104779  ...   
2016-11-02  48.744537   25.913177  106.648178   38.278000  13.396688  ...   
2016-11-03  46.812462   25.623137  103.972389   38.351501  13.430075  ...   
2016-11-04  46.192596   25.577820  104.110786   37.752499  13.366412  ...   
...               ...         ...         ...         ...        ...  ...   
2023-09-25  60.211079  134.547714  252.719193  131.270004  81.657036  ...   
2023-09-26  59.367947  132.075256  254.783630  125.980003  79.923798  ...   
2023-09-27  59.125664  133.040588  254.726837  125.980003  79.984512  ...   
2023-09-28  59.387325  136.153320  256.469299  125.980003  81.471947  ...   
2023-09-29  58.728321  136.379913  254.518509  127.120003  81.332901  ...   

Price        Volume                                                           \
Ticker           FI      FIS     INTU      LIN     LULU       MGM       MRNA   
Date                                                                           
2016-10-31  2924000  1785100  1348400  1258000  1821800   3605700        NaN   
2016-11-01  2957600  4195900  1048100   866200  1397500   9406000        NaN   
2016-11-02  2615400  2216400  1204400   957200  1849700   6798200        NaN   
2016-11-03  2009800  1101500  1110300   759300  1994000   5990800        NaN   
2016-11-04  2106000  1433400  1035000   855400  1534200  11225200        NaN   
...             ...      ...      ...      ...      ...       ...        ...   
2023-09-25  1513200  3689500  1279200  1558100  1576700   5073400  2909000.0   
2023-09-26  2147800  5478800  1233600  1769500  1521100   5338700  2888400.0   
2023-09-27  3084600  4843700  1260200  1254900   700400   5047300  2367400.0   
2023-09-28  2599700  4327300  1262700  1065500  1146500   4067200  2034500.0   
2023-09-29  2376300  6005700  1499600  1474100  1509100   3502000  4201200.0   

Price                                     
Ticker          NOW        UBER     WDAY  
Date                                      
2016-10-31  2790700         NaN  1147300  
2016-11-01  2562500         NaN  1091400  
2016-11-02  1646600         NaN  1127800  
2016-11-03  1629300         NaN   633300  
2016-11-04  1630800         NaN   892600  
...             ...         ...      ...  
2023-09-25   801400   9175400.0   894200  
2023-09-26  1509200  11283600.0  1217300  
2023-09-27  1016200  16835100.0  2338100  
2023-09-28  1158000  22199200.0  9196300  
2023-09-29  1337000  14237200.0  4066600  

[1740 rows x 930 columns]

In [15]:
# version secuencial de download_fresh_daily_prices

def download_fresh_daily_prices(data):
    stocks = data.index.get_level_values('ticker').unique().tolist()

    new_df = yf.download(tickers=stocks,
                        start=data.index.get_level_values('date').unique()[0]-pd.DateOffset(months=12),
                        end=data.index.get_level_values('date').unique()[-1],auto_adjust=False)

    return new_df
    
# start_time=time.time()

new_df=download_fresh_daily_prices(clusters_data)

# print(f'download_daily_fresh_prices sequential time: {time.time()-start_time}')



# SI SIRVIO PARALELIZAR

[*********************100%***********************]  155 of 155 completed


# Paralelizacion calculate_each_day_portfolio_return

In [16]:
@ray.remote
def calculate_return_for_date(start_date, fixed_dates, new_df, returns_dataframe):
    try:
        end_date = (pd.to_datetime(start_date)+pd.offsets.MonthEnd(0)).strftime('%Y-%m-%d')
        cols = fixed_dates[start_date]
        optimization_start_date = (pd.to_datetime(start_date)-pd.DateOffset(months=12)).strftime('%Y-%m-%d')
        optimization_end_date = (pd.to_datetime(start_date)-pd.DateOffset(days=1)).strftime('%Y-%m-%d')

        optimization_df = new_df[optimization_start_date:optimization_end_date]['Adj Close'][cols]

        try:
            weights = optimize_weights(prices=optimization_df,
                                  lower_bound=round(1/(len(optimization_df.columns)*2),3))
            weights = pd.DataFrame(weights, index=optimization_df.columns, columns=[0])
        except:
            weights = pd.DataFrame([1/len(optimization_df.columns) for _ in range(len(optimization_df.columns))],
                                   index=optimization_df.columns.tolist(),
                                   columns=[0])

        temp_df = returns_dataframe[start_date:end_date][cols]
        portfolio_returns = [
            sum(temp_df.loc[date, t] * weights.loc[t, 0] for t in temp_df.columns if pd.notna(temp_df.loc[date, t]) and t in weights.index)
            for date in temp_df.index
        ]
        return pd.DataFrame(portfolio_returns, index=temp_df.index, columns=['Strategy Return'])

    except Exception as e:
        print(f"Error for {start_date}: {e}")
        return pd.DataFrame()




# NUEVA forma: paralelizar por fecha
returns_dataframe = np.log(new_df['Adj Close']).diff()

start_time=time.time();
futures = [
    calculate_return_for_date.remote(start_date, fixed_dates, new_df, returns_dataframe)
    for start_date in fixed_dates.keys()
]
results = ray.get(futures)
portfolio_df = pd.concat(results).drop_duplicates()

print(f'ray time calculate_each_day_portfolio_return: {time.time()-start_time}')

portfolio_df

2025-06-11 21:48:02,431	INFO worker.py:1694 -- Connecting to existing Ray cluster at address: 172.18.0.2:6379...
2025-06-11 21:48:02,472	INFO worker.py:1888 -- Connected to Ray cluster.
(calculate_return_for_date pid=7083) /opt/conda/lib/python3.11/site-packages/pypfopt/expected_returns.py:56: FutureWarning: The 'fill_method' and 'limit' keywords in DataFrame.pct_change are deprecated and will be removed in a future version. Call ffill before calling pct_change instead.
(calculate_return_for_date pid=7083)   returns = prices.pct_change(fill_method=None).dropna(how="all")
(calculate_return_for_date pid=7083) /opt/conda/lib/python3.11/site-packages/pypfopt/expected_returns.py:56: FutureWarning: The 'fill_method' and 'limit' keywords in DataFrame.pct_change are deprecated and will be removed in a future version. Call ffill before calling pct_change instead.
(calculate_return_for_date pid=7083)   returns = prices.pct_change(fill_method=None).dropna(how="all")


ray time calculate_each_day_portfolio_return: 8.289974212646484


,Strategy Return
Date,
2017-11-01,NaN
2022-06-01,0.010747
2022-06-02,-0.004150
2022-06-03,0.007423
2022-06-06,0.000235
2022-06-07,0.022864
2022-06-08,-0.006346
2022-06-09,-0.023784
2022-06-10,-0.016416


## Version secuencial de calculate_each_day_portfolio_return

In [17]:
 def calculate_each_day_portfolio_return(new_df,fixed_dates):

    returns_dataframe = np.log(new_df['Adj Close']).diff()
    
    portfolio_df = pd.DataFrame()
    
    for start_date in fixed_dates.keys():
    
        try:
            end_date = (pd.to_datetime(start_date)+pd.offsets.MonthEnd(0)).strftime('%Y-%m-%d')
            cols = fixed_dates[start_date]
            optimization_start_date = (pd.to_datetime(start_date)-pd.DateOffset(months=12)).strftime('%Y-%m-%d')
            optimization_end_date = (pd.to_datetime(start_date)-pd.DateOffset(days=1)).strftime('%Y-%m-%d')
    
            optimization_df = new_df[optimization_start_date:optimization_end_date]['Adj Close'][cols]
    
            success = False
            try:
                weights = optimize_weights(prices=optimization_df,
                                       lower_bound=round(1/(len(optimization_df.columns)*2),3))
                weights = pd.DataFrame(weights, index=optimization_df.columns, columns=[0])
                success = True
            except:
                print(f'Max Sharpe Optimization failed for {start_date}, Continuing with Equal-Weights')
    
            if success==False:
                weights = pd.DataFrame([1/len(optimization_df.columns) for i in range(len(optimization_df.columns))],
                                         index=optimization_df.columns.tolist(),
                                         columns=[0])
    
            temp_df = returns_dataframe[start_date:end_date][cols]
    
            # Create portfolio returns by calculating weighted average
            portfolio_returns = []
            for date in temp_df.index:
                daily_return = 0
                for ticker in temp_df.columns:
                    if pd.notna(temp_df.loc[date, ticker]) and ticker in weights.index:
                        daily_return += temp_df.loc[date, ticker] * weights.loc[ticker, 0]
                portfolio_returns.append(daily_return)
    
            temp_portfolio_df = pd.DataFrame(portfolio_returns,
                                           index=temp_df.index,
                                           columns=['Strategy Return'])
    
            portfolio_df = pd.concat([portfolio_df, temp_portfolio_df], axis=0)
    
        except Exception as e:
            print(f"Error for {start_date}: {e}")
    
    portfolio_df = portfolio_df.drop_duplicates()
    
    return portfolio_df


start_time=time.time();
portfolio_results=calculate_each_day_portfolio_return(new_df,fixed_dates)

print(f'secuential time calculate_each_day_portfolio_return: {time.time()-start_time}')

portfolio_results

# SI SIRVIO PARALELIZAR


Max Sharpe Optimization failed for 2022-06-01, Continuing with Equal-Weights
secuential time calculate_each_day_portfolio_return: 13.682015419006348


,Strategy Return
Date,
2017-11-01,NaN
2022-06-01,0.010747
2022-06-02,-0.004150
2022-06-03,0.007423
2022-06-06,0.000235
2022-06-07,0.022864
2022-06-08,-0.006346
2022-06-09,-0.023784
2022-06-10,-0.016416
